# VocEd Lab 4 — Targeted Cleanup and Connected Components

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/Labs/Lab4_targeted_cleanup_connected_components.ipynb)

In Lab 3 we cleaned predicted masks with opening and closing — the same small disk applied to
every pixel, whatever was going wrong there.  This lab looks first, then fixes:

- *What exactly goes wrong?*  We count the separate blobs in a predicted mask with
  **connected-component labelling**, and measure how big each one is.
- *Can we fix each problem directly?*  We write two targeted steps: **remove nucleus specks**
  too small to be a real nucleus, and **fill holes** inside a nucleus.
- *Do they help, alone and together?*  An ablation, as in Lab 3, then one final test.

This is the last lab on thresholding.  Its final table is the score that the classifiers of
the next labs have to beat.

**Textbook:** [Chapter 3 — Connected Component Analysis](https://obooks.tech/imageprocessing/chapter3.html#connected-component-analysis),
[§4.6 Stage 4 — Targeted Cleanup](https://obooks.tech/imageprocessing/chapter4.html#sec-stage4), and
[Appendix C](https://obooks.tech/imageprocessing/appendixc.html) (validation).

---

### How the puzzles work 🧩🤔

- **🧩 Code puzzles:** replace every `___` and run the cell — a ✅ means you got it.
- **🤔 Think puzzles:** answer in the ✏️ cell below the question.  There is no check: discuss
  your answer with your group.

Some cells take a minute to run; they say so.

## 0. Setup — Get the Data

As in Lab 3: install `scikit-optimize`, copy the repository, load the images and masks, convert
to grayscale, and make **the same** train / validation / test split.

In [ ]:
!pip install scikit-optimize --quiet
!git clone https://github.com/emilsar/VocEd.git
%cd VocEd

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

X = np.stack([np.load(f"imagedata/X/{i}.npy") for i in range(200)])   # (200, 3, 256, 256)
y = np.stack([np.load(f"imagedata/y/{i}.npy") for i in range(200)])   # (200, 256, 256)
gray_all = 0.299 * X[:, 0] + 0.587 * X[:, 1] + 0.114 * X[:, 2]        # (200, 256, 256)

nuc_px   = (y == 2).sum(axis=(1, 2))
quartile = np.digitize(nuc_px, np.percentile(nuc_px, [25, 50, 75]))
trainval_idx, test_idx = train_test_split(np.arange(200), test_size=0.2, stratify=quartile, random_state=42)
train_idx, val_idx = train_test_split(trainval_idx, test_size=40, stratify=quartile[trainval_idx], random_state=42)

assert list(test_idx[:5]) == [121, 152, 81, 4, 141], "Not the same test images as Labs 2 and 3"
print("train:", len(train_idx), "  validation:", len(val_idx), "  test:", len(test_idx))

## 1. Recap: the Segmenter, Its Score, and the N/C Ratio

Lab 3's functions.  Lab 3's tuning found `t_nucleus ≈ 0.40` and `t_background = 1.00`, so we
use those thresholds to look at what goes wrong.

📖 Book: [§4.3 Stage 1 — Grayscale Thresholding](https://obooks.tech/imageprocessing/chapter4.html#sec-stage1)

In [ ]:
def segment(gray, t_nucleus, t_background):
    pred = np.zeros(gray.shape, dtype=np.int64)
    pred[gray < t_nucleus] = 2
    pred[(gray >= t_nucleus) & (gray < t_background)] = 1
    return pred

def dice_score(pred, target, label):
    pred_mask   = (pred   == label)
    target_mask = (target == label)
    denominator = pred_mask.sum() + target_mask.sum()
    if denominator == 0:
        return 1.0
    return 2 * (pred_mask & target_mask).sum() / denominator

def image_dice(pred, mask):
    return (dice_score(pred, mask, 1) + dice_score(pred, mask, 2)) / 2      # cytoplasm and nucleus

def nc_ratio(mask):
    return (mask == 2).sum() / max((mask == 1).sum(), 1)

pred7 = segment(gray_all[7], 0.40, 1.00)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(X[7].transpose(1, 2, 0))
axes[0].set_title('image 7')
axes[1].imshow(y[7], cmap='viridis', vmin=0, vmax=2)
axes[1].set_title('true mask')
axes[2].imshow(pred7, cmap='viridis', vmin=0, vmax=2)
axes[2].set_title(f'threshold (0.40, 1.00): Dice {image_dice(pred7, y[7]):.3f}')
for ax in axes:
    ax.axis('off')
plt.show()

Compare the two masks.  The predicted nucleus (yellow) has **specks** scattered across the
cytoplasm, and the real nuclei have lighter patches inside them that came out as cytoplasm.
How many specks are there, and how big are they?  To answer that, we need to treat each blob
as one object.

## 2. Connected Components: Counting Blobs

`label` from `skimage.measure` finds every group of touching pixels in a True/False mask and
gives each group its own number: the first blob's pixels become 1, the second's 2, and so on.
Background stays 0.  `connectivity=2` means diagonal neighbours count as touching.

📖 Book: [Chapter 3 — Connected Components and Connectivity](https://obooks.tech/imageprocessing/chapter3.html#connected-components-and-connectivity)

### 🧩 Puzzle 1 — How many nuclei?

Label the **true** nucleus mask of image 7, then the **predicted** one.

<details><summary><b>💡 Hint</b> (click to open)</summary>

The nucleus pixels of a mask are the ones equal to 2.  `return_num=True` makes `label` also return the number of blobs.

</details>

In [ ]:
from skimage.measure import label

true_labels, n_true = label(y[7] ___, connectivity=2, return_num=True)
pred_labels, n_pred = label(pred7 ___, connectivity=2, return_num=True)

print("Nuclei in the true mask:     ", n_true)
print("Nucleus blobs in the predicted mask:", n_pred)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(true_labels, cmap='nipy_spectral')
axes[0].set_title(f'true: {n_true} nuclei')
axes[1].imshow(pred_labels, cmap='nipy_spectral')
axes[1].set_title(f'predicted: {n_pred} blobs')
for ax in axes:
    ax.axis('off')
plt.show()

# ── Check ──
assert n_true == 2, "Label the true mask's nucleus pixels"
assert n_pred == 47, "Label the predicted mask's nucleus pixels"
print("✅ Puzzle 1 solved!")

### 🧩 Puzzle 2 — How big is each blob?

In the labelled image, blob 5 is all the pixels equal to 5.  `np.bincount` counts how many
times each number appears, so `np.bincount(pred_labels.ravel())[5]` is the area of blob 5.
Entry 0 counts the pixels that are *not* in any blob — we drop it.

📖 Book: [Chapter 3 — Counting Nuclei Across the Dataset](https://obooks.tech/imageprocessing/chapter3.html#counting-nuclei-across-the-dataset)

<details><summary><b>💡 Hint</b> (click to open)</summary>

`.ravel()` flattens the 2-D image to one long row.  To drop the first entry of an array, slice from position 1.

</details>

In [ ]:
pred_areas = np.bincount(pred_labels.ravel())___

print("Number of blobs:", len(pred_areas))
print("Blob areas, smallest to largest:", np.sort(pred_areas))

# The true nuclei of all 120 training images, for comparison
true_areas = []
for i in train_idx:
    areas = np.bincount(label(y[i] == 2, connectivity=2).ravel())[1:]
    true_areas.extend(areas)
true_areas = np.array(true_areas)
print(f"True nuclei in training images: {len(true_areas)}, smallest {true_areas.min()} px, "
      f"5th percentile {np.percentile(true_areas, 5):.0f} px, median {np.median(true_areas):.0f} px")

bins = np.logspace(0, 4.5, 30)
plt.hist(true_areas, bins=bins, alpha=0.6, label='true nuclei (training images)')
plt.hist(pred_areas, bins=bins, alpha=0.6, label='predicted blobs (image 7)')
plt.xscale('log')
plt.xlabel('area (pixels, log scale)')
plt.ylabel('count')
plt.legend()
plt.show()

# ── Check ──
assert len(pred_areas) == n_pred, "One area per blob — drop entry 0, the pixels outside every blob"
print("✅ Puzzle 2 solved!")

### 🤔 Think 1 — Where to cut?

1. Most predicted blobs in image 7 are tiny.  How many of them are under 100 pixels?  How does
   that compare with the size of a real nucleus?
2. We want a rule: *"a nucleus blob smaller than `min_size` pixels is a speck, not a nucleus."*
   Looking at the histogram, roughly where would you put `min_size`?  What goes wrong if it is
   far too small?  Far too large?
3. The largest predicted blob is much bigger than either true nucleus.  What might it be?

📖 Book: [§4.6.1 Diagnosing the Specific Failure Modes](https://obooks.tech/imageprocessing/chapter4.html#diagnosing-the-specific-failure-modes)

✏️ **Your answer:** *(double-click this cell and type here)*

## 3. Two Targeted Steps

### Removing specks

`remove_specks` labels the nucleus blobs, finds the ones smaller than `min_size`, and turns
their pixels into cytoplasm.  (The book does the same with `skimage`'s `remove_small_objects`;
here we write it ourselves so we can see inside.)

### 🧩 Puzzle 3 — Write `remove_specks`

📖 Book: [§4.6.2 Two Targeted Operations](https://obooks.tech/imageprocessing/chapter4.html#two-targeted-operations)

<details><summary><b>💡 Hint 1</b> (click to open)</summary>

The small blobs are those whose area is less than `min_size`.  In the mask, 1 means cytoplasm and 2 means nucleus.

</details>

<details><summary><b>💡 Hint 2</b> (click to open)</summary>

`np.isin(labels, small_ids)` is True for every pixel whose blob number is in `small_ids`.  We also require `labels > 0`, because entry 0 of `areas` is the non-nucleus pixels.

</details>

In [ ]:
def remove_specks(pred, min_size=100):
    labels    = label(pred == 2, connectivity=2)
    areas     = np.bincount(labels.ravel())
    small_ids = np.where(areas < ___)[0]
    specks    = np.isin(labels, small_ids) & (labels > 0)
    out = pred.copy()
    out[specks] = ___                 # specks become cytoplasm
    return out

after7 = remove_specks(pred7)
print(f"Nucleus blobs: before {n_pred}, after {label(after7 == 2, connectivity=2).max()}")
print(f"Dice: before {image_dice(pred7, y[7]):.4f}, after {image_dice(after7, y[7]):.4f}")

# ── Check ──
assert label(after7 == 2, connectivity=2).max() == np.sum(pred_areas >= 100), "Keep only the blobs of at least min_size pixels"
assert ((after7 != pred7) <= (pred7 == 2)).all() and (after7[after7 != pred7] == 1).all(), "Removed specks should become cytoplasm (1)"
print("✅ Puzzle 3 solved!")

### Filling holes

`binary_fill_holes` from `scipy.ndimage` takes a True/False mask and turns on every pixel that
is completely surrounded by True pixels.  Applied to the nucleus mask, it closes the lighter
patches *inside* a nucleus — and nothing outside it.

### 🧩 Puzzle 4 — Write `fill_holes`

Only cytoplasm pixels inside a hole should become nucleus.

<details><summary><b>💡 Hint</b> (click to open)</summary>

`filled` is True for the nucleus and its holes.  The pixels to change are those that are in `filled` but are labelled cytoplasm in `pred`.

</details>

In [ ]:
from scipy.ndimage import binary_fill_holes

def fill_holes(pred):
    filled = binary_fill_holes(pred == 2)
    out = pred.copy()
    out[filled & (pred == ___)] = 2      # holes become nucleus
    return out

filled7 = fill_holes(after7)
print(f"Pixels changed by fill_holes: {(filled7 != after7).sum()}")
print(f"Dice: specks removed {image_dice(after7, y[7]):.4f}, then holes filled {image_dice(filled7, y[7]):.4f}")

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (title, mask) in zip(axes, [('true mask', y[7]), ('threshold only', pred7),
                                     ('+ remove_specks', after7), ('+ fill_holes', filled7)]):
    ax.imshow(mask, cmap='viridis', vmin=0, vmax=2)
    ax.set_title(title)
    ax.axis('off')
plt.show()

# ── Check ──
assert (filled7[filled7 != after7] == 2).all() and (after7[filled7 != after7] == 1).all(), "Only cytoplasm pixels inside a hole should change, to nucleus"
print("✅ Puzzle 4 solved!")

### A third step: removing cytoplasm debris

The book also removes small **cytoplasm** blobs — debris lying on the glass, not attached to a
cell — and turns them into background.  It is the same idea, so here it is ready-made.

In [ ]:
def remove_debris(pred, min_size=100):
    labels    = label(pred == 1, connectivity=2)
    areas     = np.bincount(labels.ravel())
    small_ids = np.where(areas < min_size)[0]
    debris    = np.isin(labels, small_ids) & (labels > 0)
    out = pred.copy()
    out[debris] = 0                   # debris becomes background
    return out

print(f"Pixels changed by remove_debris on image 7: {(remove_debris(pred7) != pred7).sum()}")

### 🧩 Puzzle 5 — Are the changed pixels really what we think?

Each step changes some pixels on purpose.  Is it right more often than wrong?  Over all 120
training images, count the true label of every pixel each step changes:

- `remove_specks` changes nucleus → cytoplasm.  It is **right** where the truth is cytoplasm.
- `fill_holes` changes cytoplasm → nucleus.  It is **right** where the truth is nucleus.

*(About 10 seconds.)*

<details><summary><b>💡 Hint</b> (click to open)</summary>

`y[i][changed]` is the true label of each changed pixel.  Count the ones equal to the label that makes the change correct.

</details>

In [ ]:
specks_changed, specks_right = 0, 0
fill_changed,   fill_right   = 0, 0

for i in train_idx:
    pred = segment(gray_all[i], 0.40, 1.00)

    changed = remove_specks(pred) != pred
    specks_changed += changed.sum()
    specks_right   += (y[i][changed] == ___).sum()

    changed = fill_holes(pred) != pred
    fill_changed += changed.sum()
    fill_right   += (y[i][changed] == ___).sum()

print(f"remove_specks changed {specks_changed:,} pixels; right on {specks_right / specks_changed:.0%}")
print(f"fill_holes    changed {fill_changed:,} pixels; right on {fill_right / fill_changed:.0%}")

# ── Check ──
assert specks_right > specks_changed / 2, "remove_specks is right where the truth is cytoplasm"
assert fill_right   > fill_changed / 2,   "fill_holes is right where the truth is nucleus"
print("✅ Puzzle 5 solved!")

### 🤔 Think 2 — When filling a hole is wrong

`fill_holes` is right on most of the pixels it changes, but far from all.  Here is a training
image where it goes badly wrong.

In [ ]:
pred167 = segment(gray_all[167], 0.40, 1.00)
fill167 = fill_holes(pred167)
changed = fill167 != pred167
print(f"Image 167: fill_holes changed {changed.sum()} pixels; truly nucleus: {(y[167][changed] == 2).sum()}")
print(f"N/C ratio: true {nc_ratio(y[167]):.2f}, threshold {nc_ratio(pred167):.2f}, after fill_holes {nc_ratio(fill167):.2f}")

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(X[167].transpose(1, 2, 0))
axes[0].set_title('image 167')
for ax, (title, mask) in zip(axes[1:], [('true mask', y[167]), ('threshold only', pred167), ('+ fill_holes', fill167)]):
    ax.imshow(mask, cmap='viridis', vmin=0, vmax=2)
    ax.set_title(title)
for ax in axes:
    ax.axis('off')
plt.show()

1. Look at where the changed pixels are.  What "hole" did `fill_holes` fill here, and what was
   really in it?
2. `fill_holes` only asks *"is this region surrounded by nucleus?"*.  Why is that question
   safe for a single round nucleus, but not when dark pixels form a **ring**?
3. Look at the N/C ratios.  A few thousand wrong pixels on one image — why can that matter more
   for the N/C ratio than for the average Dice?

📖 Book: [Chapter 3 — Connected Components and Connectivity](https://obooks.tech/imageprocessing/chapter3.html#connected-components-and-connectivity)

✏️ **Your answer:** *(double-click this cell and type here)*

## 4. The Ablation Study

As in Lab 3, a pipeline is a list of mask steps, and every pipeline gets its **own** thresholds,
re-tuned on the training images by Bayesian optimisation.  We add Lab 3's opening and closing
(radius 2) for comparison.  *(About a minute.)*

📖 Book: [Appendix B — Practical Implementation with Scikit-Optimize](https://obooks.tech/imageprocessing/appendixb.html#practical-implementation-with-scikit-optimize)

In [ ]:
from skopt import gp_minimize
from skimage import morphology

def open_nucleus(pred, radius=2):
    nucleus = (pred == 2)
    kept = morphology.opening(nucleus, morphology.disk(radius))
    out = pred.copy()
    out[nucleus & ~kept] = 1
    return out

def close_nucleus(pred, radius=2):
    nucleus = (pred == 2)
    filled = morphology.closing(nucleus, morphology.disk(radius))
    out = pred.copy()
    out[filled & (pred == 1)] = 2
    return out

def image_scores(idx, t_nucleus, t_background, mask_steps):
    scores = []
    for i in idx:
        pred = segment(gray_all[i], t_nucleus, t_background)
        for step in mask_steps:
            pred = step(pred)
        scores.append(image_dice(pred, y[i]))
    return np.array(scores)

def evaluate(name, mask_steps):
    def objective(params):
        return -image_scores(train_idx, params[0], params[1], mask_steps).mean()

    result = gp_minimize(objective, [(0.20, 0.60), (0.80, 1.00)], n_calls=20, random_state=0)
    val = image_scores(val_idx, result.x[0], result.x[1], mask_steps)
    print(f"{name:22s} thresholds ({result.x[0]:.3f}, {result.x[1]:.2f})   train {-result.fun:.4f}   validation {val.mean():.4f}")
    return val, result.x

pipelines = {
    'none':                 [],
    'specks':               [remove_specks],
    'fill':                 [fill_holes],
    'debris':               [remove_debris],
    'specks, fill':         [remove_specks, fill_holes],
    'fill, specks':         [fill_holes, remove_specks],
    'specks, fill, debris': [remove_specks, fill_holes, remove_debris],
    'open, close (Lab 3)':  [open_nucleus, close_nucleus],
}

val_scores, thresholds = {}, {}
for name, mask_steps in pipelines.items():
    val_scores[name], thresholds[name] = evaluate(name, mask_steps)

### 🤔 Think 3 — Together is more than the sum

1. On validation, how much does `specks` alone add to `none`?  `fill` alone?  And
   `specks, fill` together?  Is the gain of the pair bigger than the two gains added up?
2. Look at the thresholds.  With `specks, fill`, `t_nucleus` drops from about 0.40 to about
   0.35 — fewer pixels are called nucleus at first.  Why can the pair afford a stricter
   threshold, when neither step can alone?  *(Think about what each step repairs when the
   threshold is stricter.)*
3. `specks, fill` and `fill, specks` score almost the same.  In Lab 3, the order of
   `blur, sharpen` mattered.  Why does the order hardly matter here?

📖 Book: [§4.6 Stage 4 — Targeted Cleanup](https://obooks.tech/imageprocessing/chapter4.html#sec-stage4)

✏️ **Your answer:** *(double-click this cell and type here)*

### 🤔 Think 4 — The debris that wasn't

`remove_debris` changes the validation Dice by almost nothing, and adding it to `specks, fill`
does not help either.  The book's Stage 4 removes debris — so why does it do so little here?

1. Section 3 printed how many pixels `remove_debris` changes on image 7.  Is it a lot?
2. With `t_background = 1.00`, which pixels count as background?  How likely is a small
   island of cytoplasm, entirely surrounded by background, to appear?
3. What does this tell you about copying a fix from somewhere else without measuring it on
   your own data and settings?

📖 Book: [§4.6.3 Why Size Beats Proximity](https://obooks.tech/imageprocessing/chapter4.html#why-size-beats-proximity)

✏️ **Your answer:** *(double-click this cell and type here)*

### 🧩 Puzzle 6 — Image by image

Lab 3 taught us to look past the average: on how many of the 40 validation images does
`specks, fill` beat `none`?

<details><summary><b>💡 Hint</b> (click to open)</summary>

The comparison gives an array of True/False.  Count the Trues the same way as in Lab 3, Puzzle 6.

</details>

In [ ]:
better = (val_scores['specks, fill'] > val_scores['none']).___()
worse  = (val_scores['specks, fill'] < val_scores['none']).___()
print(f"specks, fill beats none on {better} of 40 validation images, loses on {worse}")

plt.scatter(val_scores['none'], val_scores['specks, fill'], alpha=0.7)
plt.plot([0.4, 1], [0.4, 1], 'r--')
plt.xlabel("Dice, threshold only")
plt.ylabel("Dice, specks + fill")
plt.title("Each dot is one validation image")
plt.show()

# ── Check ──
assert better == np.sum(val_scores['specks, fill'] > val_scores['none']), "Count the True values"
assert better + worse <= 40, "Count images, not pixels"
print("✅ Puzzle 6 solved!")

## 5. Three Knobs at Once

In the ablation `min_size` was fixed at 100.  Let's give Bayesian optimisation three knobs —
both thresholds and `min_size` — for the `specks, fill` pipeline.  *(About 15 seconds.)*

### 🧩 Puzzle 7 — The search space

`min_size` is a whole number of pixels, so it uses `Integer`.  Let it range from 0 (no speck
removal) to 1000.

📖 Book: [Appendix B — Practical Implementation with Scikit-Optimize](https://obooks.tech/imageprocessing/appendixb.html#practical-implementation-with-scikit-optimize)

In [ ]:
from skopt.space import Real, Integer

def predict(gray, t_nucleus, t_background, min_size):
    pred = segment(gray, t_nucleus, t_background)
    pred = remove_specks(pred, min_size)
    return fill_holes(pred)

def pipeline_dice(idx, params):
    return np.mean([image_dice(predict(gray_all[i], *params), y[i]) for i in idx])

space = [Real(0.20, 0.60),     # t_nucleus
         Real(0.80, 1.00),     # t_background
         ___]   # min_size

result3 = gp_minimize(lambda params: -pipeline_dice(train_idx, params), space, n_calls=30, random_state=0)
best3 = result3.x
print(f"Best settings: t_nucleus {best3[0]:.3f}, t_background {best3[1]:.2f}, min_size {best3[2]}")
print(f"Training Dice:   {-result3.fun:.4f}")
print(f"Validation Dice: {pipeline_dice(val_idx, best3):.4f}")

# ── Check ──
assert len(space) == 3 and space[2].low == 0 and space[2].high == 1000, "min_size: whole numbers from 0 to 1000"
print("✅ Puzzle 7 solved!")

### 🤔 Think 5 — More freedom, worse validation

Compare with the ablation's `specks, fill` row, where `min_size` was simply fixed at 100.

1. Which has the higher **training** Dice?  Which has the higher **validation** Dice?
2. The optimiser chose a `min_size` near 200.  Look back at the histogram in Puzzle 2: what
   kind of nucleus does a cut at 200 start to erase?
3. We now have two candidates.  Which one should go to the final test, and on what evidence?

📖 Book: [Appendix C — Overfitting and Underfitting](https://obooks.tech/imageprocessing/appendixc.html#overfitting-and-underfitting)

✏️ **Your answer:** *(double-click this cell and type here)*

## 6. The Final Test — Once

Our choice, by validation: `specks, fill` with `min_size = 100`.  We score it on the 40 test
images once, next to thresholding alone and the three-knob search, and add the **fraction of
test images whose N/C ratio is within ±0.05** of the truth — a number a pathologist could read.

📖 Book: [§4.7 Comparing All Four Stages](https://obooks.tech/imageprocessing/chapter4.html#comparing-all-four-stages)

In [ ]:
nc_true = np.array([nc_ratio(y[i]) for i in test_idx])
t_nuc, t_bg = thresholds['specks, fill']

candidates = {
    'threshold only (0.40, 1.00)':   lambda gray: segment(gray, 0.40, 1.00),
    'specks, fill (our choice)':     lambda gray: predict(gray, t_nuc, t_bg, 100),
    'three-knob search':             lambda gray: predict(gray, *best3),
}

print(f"{'':30s} test Dice   N/C R²   N/C within ±0.05")
for name, model in candidates.items():
    preds   = [model(gray_all[i]) for i in test_idx]
    dice    = np.mean([image_dice(p, y[i]) for p, i in zip(preds, test_idx)])
    nc_pred = np.array([nc_ratio(p) for p in preds])
    r2      = 1 - np.sum((nc_pred - nc_true) ** 2) / np.sum((nc_true - nc_true.mean()) ** 2)
    close   = np.mean(np.abs(nc_pred - nc_true) < 0.05)
    print(f"{name:30s} {dice:.4f}    {r2:7.3f}   {close:.0%}")

print()
print("Earlier labs, on the same 40 test images:")
print("  Lab 2 (0.40, 0.85, blur σ=2)    0.719")
print("  Lab 3 five-knob pipeline        0.770     -0.812")

### 🤔 Think 6 — The end of thresholding

1. Did `specks, fill` keep its lead over thresholding alone on the test images?  How does it
   compare with Lab 3's five-knob pipeline — and which of the two would you rather explain to
   someone?
2. Look at the N/C columns.  Is the N/C ratio usable yet?  On what share of images is it
   within ±0.05?
3. Four labs of tuning moved test Dice from about 0.72 to about 0.77.  Every method so far
   decides from **one brightness value per pixel**, and then tidies the result.  Name one thing
   a pathologist uses to tell nucleus from cytoplasm that a single brightness value cannot carry.

📖 Book: [§4.7.2 The Ceiling, and What Comes Next](https://obooks.tech/imageprocessing/chapter4.html#the-ceiling-and-what-comes-next)

✏️ **Your answer:** *(double-click this cell and type here)*

## Wrap-up

**Key takeaways:**
- **Connected-component labelling** turns a mask into objects: count them with `label`, size
  them with `np.bincount`.  Looking at the objects told us what to fix.
- **Targeted steps** fix one diagnosed problem each: `remove_specks` for blobs too small to be a
  nucleus, `fill_holes` for lighter patches inside one.  Measure how often each change is right.
- **Steps interact.**  Together, and with re-tuned thresholds, the two steps gained twice what
  they gained alone.  The debris step from the book gained nothing on our settings.
- **More knobs can fit the training images better and the validation images worse.**  Choose
  on validation.
- **The ceiling holds.**  Test Dice ≈ 0.77 and an N/C ratio that is not yet usable.  From Lab 5
  on, each pixel gets more than one brightness value to go on, and a classifier learns the rule.

---

### Group discussion 👥

Split the work, then share your numbers:

| Person | Change, then re-run Section 4 |
|---|---|
| A | `remove_specks(pred, min_size=50)`: change the default to 50 |
| B | `remove_specks(pred, min_size=200)`: change the default to 200 |
| C | Add `'specks, fill, open': [remove_specks, fill_holes, open_nucleus]` to `pipelines` |

1. How sensitive is the `specks, fill` row to `min_size`?  Does that match Think 5?
2. Did adding opening to the targeted steps help, or do they already do its job?
3. If you could add one more targeted step for the one big merged blob in image 7, what would it do?